In [19]:
import numpy as np
import pandas as pd
import os

file_path = "interactions_clean.csv"

# UNCOMMENT THIS IF YOU ARE USING KAGGLE
#file_path = '/kaggle/input/datasets/djamelachour/student-question-interaction-data/interactions_clean.csv'

df = pd.read_csv('interactions_clean.csv')


df.head()

,Student ID,Question ID,difficulty,correct,Response Time,topic
0,AQklHbi5bt8l,1,hard,0,27232,Data Mining Tasks
1,AQklHbi5bt8l,2,easy,1,8933,Similarity Measures
2,AQklHbi5bt8l,3,easy,1,14965,Similarity Measures
3,AQklHbi5bt8l,4,easy,1,12691,Similarity Measures
4,AQklHbi5bt8l,5,medium,0,26452,Similarity Measures


In [20]:
df.shape

(17340, 6)

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17340 entries, 0 to 17339
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Student ID     17340 non-null  object
 1   Question ID    17340 non-null  int64 
 2   difficulty     17340 non-null  object
 3   correct        17340 non-null  int64 
 4   Response Time  17340 non-null  int64 
 5   topic          17340 non-null  object
dtypes: int64(3), object(3)
memory usage: 812.9+ KB


### Encoding categorical features as numbers

In [22]:
from sklearn.preprocessing import LabelEncoder

le_student = LabelEncoder()

df['Student ID'] = le_student.fit_transform(df['Student ID'])

In [23]:
le_topic = LabelEncoder()

df['topic'] = le_topic.fit_transform(df['topic'])

In [24]:
le_diff = LabelEncoder()

df['difficulty'] = le_diff.fit_transform(df['difficulty'])

In [25]:
df.head()

,Student ID,Question ID,difficulty,correct,Response Time,topic
0,15,1,1,0,27232,6
1,15,2,0,1,8933,22
2,15,3,0,1,14965,22
3,15,4,0,1,12691,22
4,15,5,2,0,26452,22


### Data Exploration

In [26]:
print('number of students: ', df['Student ID'].nunique())
print('number of questions: ', df['Question ID'].nunique())

number of students:  121
number of questions:  150


### label feature balance check

In [27]:
df['correct'].value_counts()

correct
1    13134
0     4206
Name: count, dtype: int64

### Data Splitting

In [28]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['correct'])
y = df['correct']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [29]:
print(f"Training set size: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test set size: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")
print(f"Training - Class distribution:\n{y_train.value_counts(normalize=True).mul(100).round(1)}")
print(f"Test - Class distribution:\n{y_test.value_counts(normalize=True).mul(100).round(1)}")

Training set size: 13872 (80.0%)
Test set size: 3468 (20.0%)
Training - Class distribution:
correct
1    75.7
0    24.3
Name: proportion, dtype: float64
Test - Class distribution:
correct
1    75.7
0    24.3
Name: proportion, dtype: float64


In [30]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ============ TRAINING SCRIPT ============
print("="*50)
print("TRAINING PROBABILISTIC CLASSIFIERS")
print("="*50)


# Initialize models
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Naive Bayes': GaussianNB()
}

# Train models
trained_models = {}
for name, model in models.items():
    print(f"\nTraining {name}...")
    trained_models[name] = model.fit(X_train, y_train)
    print(f"{name} trained successfully!")

# ============ EVALUATION SCRIPT ============
print("\n" + "="*50)
print("EVALUATION WITH PREDICTION CERTAINTY")
print("="*50)

results = {}

for name, model in trained_models.items():
    print(f"\n{'='*40}")
    print(f"MODEL: {name}")
    print('='*40)
    
    # Predict classes and probabilities
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    
    # Get max probability (certainty) for each prediction
    certainty = np.max(y_proba, axis=1)
    avg_certainty = certainty.mean() * 100
    
    # Evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Average prediction certainty: {avg_certainty:.2f}%")
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred))
    
    # Store results
    results[name] = {
        'accuracy': accuracy,
        'avg_certainty': avg_certainty,
        'y_pred': y_pred,
        'y_proba': y_proba,
        'certainty': certainty
    }
    
    # Show first 10 predictions with certainty
    print(f"\nFirst 10 predictions with certainty:")
    for i in range(min(10, len(y_test))):
        pred_class = y_pred[i]
        true_class = y_test.iloc[i] if hasattr(y_test, 'iloc') else y_test[i]
        cert = certainty[i] * 100
        print(f"  Sample {i+1}: Predicted={pred_class}, Actual={true_class}, Certainty={cert:.1f}%")

# ============ COMPARISON SUMMARY ============
print("\n" + "="*50)
print("MODEL COMPARISON SUMMARY")
print("="*50)
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['accuracy'] for m in results],
    'Avg Certainty (%)': [results[m]['avg_certainty'] for m in results]
})
print(comparison_df.to_string(index=False))

# Best model by accuracy
best_model = comparison_df.loc[comparison_df['Accuracy'].idxmax(), 'Model']
print(f"\n🏆 Best model by accuracy: {best_model}")

# Function to get certainty for a single prediction
def predict_with_certainty(model, sample):
    proba = model.predict_proba([sample])[0]
    pred = model.predict([sample])[0]
    certainty = max(proba) * 100
    return pred, certainty, proba

print("\n✅ Ready to use predict_with_certainty(model, sample)")

TRAINING PROBABILISTIC CLASSIFIERS

Training Random Forest...


Random Forest trained successfully!

Training Logistic Regression...
Logistic Regression trained successfully!

Training Naive Bayes...
Naive Bayes trained successfully!

EVALUATION WITH PREDICTION CERTAINTY

MODEL: Random Forest
Accuracy: 0.8959
Average prediction certainty: 89.50%

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.73      0.77       841
           1       0.92      0.95      0.93      2627

    accuracy                           0.90      3468
   macro avg       0.87      0.84      0.85      3468
weighted avg       0.89      0.90      0.89      3468


First 10 predictions with certainty:
  Sample 1: Predicted=1, Actual=0, Certainty=88.0%
  Sample 2: Predicted=1, Actual=1, Certainty=100.0%
  Sample 3: Predicted=1, Actual=1, Certainty=100.0%
  Sample 4: Predicted=1, Actual=1, Certainty=100.0%
  Sample 5: Predicted=1, Actual=1, Certainty=98.0%
  Sample 6: Predicted=1, Actual=1, Certainty=93.0%
  Sample 7: Predic